In [ ]:
# =====================================================================
# EXECUTION SCRIPT & KAGGLE DATALOADER (Complete Pipeline)
# =====================================================================
import os
import sys
import math
import numpy as np
import cupy as cp
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.utils as vutils
from torch.utils.data import DataLoader
from tqdm import tqdm
from numba import cuda
from safetensors.torch import save_file, load_file

# =====================================================================
# KERNEL 2: CONVOLUTIONAL KARL GENERATIVE UPDATE (Decoder/Predictive)
# =====================================================================
@cuda.jit
def conv_karl_update_kernel(G, z_current, x_true, x_pred, lr, weight_decay):
    in_c_z, out_c_x, kh = cuda.grid(3)
    if in_c_z < G.shape[0] and out_c_x < G.shape[1] and kh < G.shape[2]:
        for kw in range(G.shape[3]):
            batch_size = z_current.shape[0]
            out_h = x_true.shape[2] 
            out_w = x_true.shape[3]
            delta = 0.0
            for b in range(batch_size):
                for oh in range(out_h):
                    for ow in range(out_w):
                        error = x_true[b, out_c_x, oh, ow] - x_pred[b, out_c_x, oh, ow]
                        zh = oh - kh
                        zw = ow - kw
                        if zh >= 0 and zw >= 0 and zh < z_current.shape[2] and zw < z_current.shape[3]:
                            delta += error * z_current[b, in_c_z, zh, zw]
            
            # Apply L2 Weight decay regularization to prevent color divergence
            current_weight = G[in_c_z, out_c_x, kh, kw]
            G[in_c_z, out_c_x, kh, kw] = current_weight * (1.0 - weight_decay) + (lr / (batch_size * out_h * out_w)) * delta


# =====================================================================
# THE ZERO-COPY DLPack BRIDGE
# =====================================================================
def cupy_to_torch(cp_arr):
    """ Passes VRAM pointer from CuPy to PyTorch without copying data """
    return torch.from_dlpack(cp_arr)

def torch_to_cupy(th_tensor):
    """ Passes VRAM pointer from PyTorch back to CuPy """
    return cp.from_dlpack(th_tensor)


# =====================================================================
# COMPLETED: CONVOLUTIONAL FORWARD-FORWARD LAYER
# =====================================================================
class CUDAConvFFLayer:
    def __init__(self, in_channels, out_channels, kernel_size=3, lr_rep=0.01, lr_gen=0.01):
        self.in_c = in_channels
        self.out_c = out_channels
        self.ks = kernel_size
        self.lr_rep = lr_rep
        self.lr_gen = lr_gen

        # Encoder weights (Conv)
        limit_w = np.sqrt(6 / (in_channels * kernel_size * kernel_size + out_channels))
        self.W = cp.random.uniform(-limit_w, limit_w, (out_channels, in_channels, kernel_size, kernel_size), dtype=cp.float32)

        # Decoder weights (Transposed Conv)
        self.G = cp.random.uniform(-(limit_w * 2.0), (limit_w * 2.0), (out_channels, in_channels, kernel_size, kernel_size), dtype=cp.float32)

    def forward_encoder(self, x_cupy):
        """ cuDNN Forward Pass (Autograd OFF) """
        x_th = cupy_to_torch(x_cupy)
        w_th = cupy_to_torch(self.W)
        with torch.no_grad():
            y_th = F.conv2d(x_th, w_th, stride=1, padding=1)
            y_th = F.leaky_relu(y_th, 0.01)
        return torch_to_cupy(y_th)

    def forward_decoder(self, z_cupy):
        """ cuDNN Transposed Forward Pass (Autograd OFF) """
        z_th = cupy_to_torch(z_cupy)
        g_th = cupy_to_torch(self.G)
        with torch.no_grad():
            x_pred_th = F.conv_transpose2d(z_th, g_th, stride=1, padding=1)
            x_pred_th = torch.sigmoid(x_pred_th)  # Soft boundary applied
        return torch_to_cupy(x_pred_th)

    def train_encoder(self, x_pos, x_neg, threshold=-0.05):
        """Reconstruction-Guided Forward-Forward"""
        x_pos_th = cupy_to_torch(x_pos)
        x_neg_th = cupy_to_torch(x_neg)
        
        # Enable grad on encoder weights only
        W_th = cupy_to_torch(self.W)
        W_th.requires_grad = True
        G_th = cupy_to_torch(self.G).detach()

        # --- POSITIVE: real image should reconstruct well ---
        z_pos = F.conv2d(x_pos_th, W_th, stride=1, padding=1)
        z_pos = F.leaky_relu(z_pos, 0.01)
        x_hat_pos = F.conv_transpose2d(z_pos, G_th, stride=1, padding=1)
        x_hat_pos = torch.sigmoid(x_hat_pos)  # Soft boundary applied
        mse_pos = F.mse_loss(x_hat_pos, x_pos_th)

        # --- NEGATIVE: corrupted image should reconstruct poorly ---
        z_neg = F.conv2d(x_neg_th, W_th, stride=1, padding=1)
        z_neg = F.leaky_relu(z_neg, 0.01)
        x_hat_neg = F.conv_transpose2d(z_neg, G_th, stride=1, padding=1)
        x_hat_neg = torch.sigmoid(x_hat_neg)  # Soft boundary applied
        mse_neg = F.mse_loss(x_hat_neg, x_neg_th)

        # Goodness = negative MSE
        g_pos = -mse_pos
        g_neg = -mse_neg

        # FF margin loss
        ff_loss = torch.log(1.0 + torch.exp(g_neg - g_pos))
        loss = ff_loss + 0.5 * mse_pos
        loss.backward()

        # Copy gradient back to CuPy and update
        with torch.no_grad():
            grad_cp = torch_to_cupy(W_th.grad)
            self.W -= self.lr_rep * grad_cp

        return torch_to_cupy(z_pos.detach()), torch_to_cupy(z_neg.detach()), torch_to_cupy(ff_loss.detach())

    def train_decoder(self, z_current, x_true):
        x_pred = self.forward_decoder(z_current)
        
        threads_per_block = (8, 4, 1)
        blocks_x = math.ceil(self.out_c / threads_per_block[0])
        blocks_y = math.ceil(self.in_c / threads_per_block[1])
        blocks_z = math.ceil(self.ks / threads_per_block[2])

        weight_decay = 1e-4  # Regularizer for kernel update
        conv_karl_update_kernel[(blocks_x, blocks_y, blocks_z), threads_per_block](
            self.G, z_current, x_true, x_pred, self.lr_gen, weight_decay
        )

        mse = cp.mean((x_true - x_pred)**2)
        return mse, x_pred


# =====================================================================
# THE GENERATOR ORCHESTRATOR & SAFETENSORS MANAGER
# =====================================================================
class AnimeForwardForwardGenerator:
    def __init__(self):
        # 64x64 RGB input
        self.enc_layer1 = CUDAConvFFLayer(in_channels=3, out_channels=32, kernel_size=3)
        self.enc_layer2 = CUDAConvFFLayer(in_channels=32, out_channels=64, kernel_size=3)
        self.enc_layer3 = CUDAConvFFLayer(in_channels=64, out_channels=128, kernel_size=3)

    def train_step(self, x_pos, x_neg):
        # 1. ENCODER PASS
        z1_pos, z1_neg, ff1 = self.enc_layer1.train_encoder(x_pos, x_neg)
        z2_pos, z2_neg, ff2 = self.enc_layer2.train_encoder(z1_pos, z1_neg)
        z3_pos, z3_neg, ff3 = self.enc_layer3.train_encoder(z2_pos, z2_neg)

        latent_z = z3_pos

        # 2. DECODER PASS
        mse3, z2_pred = self.enc_layer3.train_decoder(z_current=latent_z, x_true=z2_pos)
        mse2, z1_pred = self.enc_layer2.train_decoder(z_current=z2_pos, x_true=z1_pos)
        mse1, x_pred  = self.enc_layer1.train_decoder(z_current=z1_pos, x_true=x_pos)

        total_mse = mse1 + mse2 + mse3
        total_ff = ff1 + ff2 + ff3
        return x_pred, total_mse, total_ff

    def save_safetensors(self, filepath):
        tensors = {
            "layer1.W": torch.from_numpy(self.enc_layer1.W.get()),
            "layer1.G": torch.from_numpy(self.enc_layer1.G.get()),
            "layer2.W": torch.from_numpy(self.enc_layer2.W.get()),
            "layer2.G": torch.from_numpy(self.enc_layer2.G.get()),
            "layer3.W": torch.from_numpy(self.enc_layer3.W.get()),
            "layer3.G": torch.from_numpy(self.enc_layer3.G.get())
        }
        save_file(tensors, filepath)
        print(f"\nModel weights successfully saved to {filepath}")

    def load_safetensors(self, filepath):
        tensors = load_file(filepath)
        self.enc_layer1.W = cp.array(tensors["layer1.W"].numpy())
        self.enc_layer1.G = cp.array(tensors["layer1.G"].numpy())
        self.enc_layer2.W = cp.array(tensors["layer2.W"].numpy())
        self.enc_layer2.G = cp.array(tensors["layer2.G"].numpy())
        self.enc_layer3.W = cp.array(tensors["layer3.W"].numpy())
        self.enc_layer3.G = cp.array(tensors["layer3.G"].numpy())
        print(f"Model weights successfully loaded from {filepath}. Resuming training...")


# =====================================================================
# NEGATIVE DATA GENERATOR (Patch Scrambling)
# =====================================================================
def make_negative(x, noise_scale=0.15):
    noise = np.random.normal(loc=0.0, scale=noise_scale, size=x.shape).astype(np.float32)
    x_rolled = np.roll(x, shift=2, axis=(2, 3))
    x_neg = np.clip(x_rolled + noise, 0.0, 1.0)
    return x_neg


# =====================================================================
# EXECUTION SCRIPT
# =====================================================================
if __name__ == "__main__":
    # Using custom FlatImageFolder so we don't need class subdirectories
    import glob
    from PIL import Image
    from torch.utils.data import Dataset

    class FlatImageFolder(Dataset):
        """Loads all images in a directory, ignoring class subfolders."""
        def __init__(self, root, transform=None):
            self.root = root
            self.transform = transform
            self.image_paths = (
                glob.glob(os.path.join(root, '**', '*.jpg'), recursive=True) + 
                glob.glob(os.path.join(root, '**', '*.png'), recursive=True) + 
                glob.glob(os.path.join(root, '**', '*.jpeg'), recursive=True)
            )
            if len(self.image_paths) == 0:
                raise RuntimeError(f"No images found in {root}")

        def __len__(self):
            return len(self.image_paths)

        def __getitem__(self, idx):
            img_path = self.image_paths[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (64, 64))
            if self.transform:
                image = self.transform(image)
            return image, 0

    ganyu_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    print("Loading Ganyu Dataset...")
    
    dataset_path = '/kaggle/input/datasets/andy8744/ganyu-genshin-impact-anime-faces-gan-training/ganyu'
    
    train_dataset = FlatImageFolder(root=dataset_path, transform=ganyu_transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = AnimeForwardForwardGenerator()
    
    model_path = "ganyu_ff_conv.safetensors"
    if os.path.exists(model_path):
        model.load_safetensors(model_path)

    print("Initiating Hybrid cuDNN + Numba Forward-Forward Training...")

    EPOCHS = 20
    for epoch in range(EPOCHS):
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for batch_idx, (data, _) in enumerate(train_bar):
            x_pos = cp.array(data.numpy())
            
            x_neg_np = make_negative(x_pos.get())
            x_neg = cp.array(x_neg_np)
            
            x_pred, gen_loss, ff_loss = model.train_step(x_pos, x_neg)
            train_bar.set_postfix({'Gen Loss': f'{gen_loss.get():.4f}', 'FF Loss': f'{ff_loss.get():.4f}'})
            
        print("\nGenerating Hallucinated Ganyu Reconstructions...")
        orig_img = torch.from_numpy(x_pos[:16].get())
        recon_img = torch.from_numpy(x_pred[:16].get())
        
        # We can still clamp the final render for visualization just in case, but the network output is now bounded via Sigmoid
        recon_img = torch.clamp(recon_img, 0.0, 1.0)
        
        combined = torch.cat([orig_img, recon_img], dim=0)
        img_path = f'ganyu_reconstruction_epoch_{epoch+1}.png'
        
        vutils.save_image(combined, img_path, nrow=16, normalize=False)
        print(f"Saved Reconstructions to: {img_path}")

    model.save_safetensors("ganyu_ff_conv.safetensors")

